<a href="https://colab.research.google.com/github/Mhmd-Seyed/ecommerce-behavior-analytics/blob/main/01_data_prep_ipynb_adl%C4%B1_not_defterinin_kopyas%C4%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Ortak proje klasörünün bağlı olduğunu doğrulama
!ls -la /content/drive/MyDrive/ecommerce_project

In [ ]:
!pip install -q pyspark kaggle
from google.colab import files
files.upload()  # Bilgisayarınızdan kaggle.json dosyasını seçin
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Sadece 2019-Oct.csv dosyasını indirir ve disk dolmaması için zip dosyasını siler
!kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi
category-store -f 2019-Oct.csv
!unzip -q 2019-Oct.csv.zip
!rm 2019-Oct.csv.zip
!ls -lh 2019-Oct.csv

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,
DoubleType
# 1. PySpark oturumunu başlat (Colab RAM kapasitesine göre optimize edilmiş)
spark = SparkSession.builder \
    .appName("Student1_DataPrep_And_EventModeling") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()
# 2. Okuma hızını artırmak için sütun yapısını önceden tanımla
raw_schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("category_id", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("user_session", StringType(), True)
])
df_raw = spark.read.csv("2019-Oct.csv", header=True, schema=raw_schema)
print("Ham veri satır sayısı:", df_raw.count())
# 3. Veri Temizleme: Kritik alanları boş olan, geçersiz olay türü içeren veya
fiyatı <= 0 olan satırları çıkar
df_clean = df_raw.drop("category_id") \
    .filter(F.col("user_id").isNotNull()) \
    .filter(F.col("user_session").isNotNull()) \
    .filter(F.col("event_type").isin("view", "cart", "purchase")) \
    .filter(F.col("price") > 0) \
    .dropDuplicates()
# Eksik kategori ve marka isimlerini 'unknown' olarak doldur
df_clean = df_clean.fillna({"category_code": "unknown", "brand": "unknown"})
# 4. Zaman damgası dönüşümü ("2019-10-01 00:00:00 UTC" -> Timestamp)
df_clean = df_clean.withColumn(
    "event_time",
    F.to_timestamp(F.regexp_replace("event_time", " UTC", ""), "yyyy-MM-dd
HH:mm:ss")
)

In [ ]:
# 5. Olay Türlerini Modelleme (Hiyerarşik Adım + İkili Göstergeler + Kategori/Saat
Kırılımı)
df_modeled = df_clean \
    .withColumn("event_step",
        F.when(F.col("event_type") == "view", 1)
         .when(F.col("event_type") == "cart", 2)
         .when(F.col("event_type") == "purchase", 3)
         .otherwise(0)
# 1. Ekibin hızlı testi için 50.000 satırlık örnek dosya üret
sample_df = df_modeled.limit(50000)
sample_df.write.mode("overwrite").parquet("/content/sample_events.parquet")
# 2. Tam temizlenmiş veri setini Parquet olarak kaydet
df_modeled.write.mode("overwrite").parquet("/content/events_clean.parquet")
# 3. Her iki klasörü ortak Google Drive klasörüne kopyala
!rm -rf /content/drive/MyDrive/ecommerce_project/sample_events.parquet
!rm -rf /content/drive/MyDrive/ecommerce_project/events_clean.parquet
!cp -r /content/sample_events.parquet /content/drive/MyDrive/ecommerce_project/
!cp -r /content/events_clean.parquet /content/drive/MyDrive/ecommerce_project/
print("Tüm dosyalar başarıyla ortak Google Drive klasörüne aktarıldı!")